# Component 2 – Drift model experiments
Goal: predict drift distance (m) from six inputs known before spraying.
Inputs: crosswind, height, swath, temperature, RH, droplet class. Target: drift_distance_m.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split

DATA = Path.cwd().parent / "data" / "agdrift_dataset_v3_final.csv"
df = pd.read_csv(DATA)

DROPLET_CODE = {"VF-F": 0, "F-M": 1, "M-C": 2, "C-VC": 3}
df["droplet_code"] = df["droplet_class"].map(DROPLET_CODE)

FEATURES = ["crosswind_ms", "height_m", "swath_m", "temp_c", "rh_pct", "droplet_code"]
TARGET = "drift_distance_m"
SEED = 42
print(df.shape)

(5000, 18)


## M1 – Train/test split
80 % of runs are used to train the model, 20 % are kept aside to test it on unseen data.
The random seed makes the split repeatable.

In [2]:
train, test = train_test_split(df, test_size=0.2, random_state=SEED)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (4000, 6) | Test: (1000, 6)


In [3]:
cols = FEATURES + [TARGET]
ks = pd.DataFrame([[r.statistic, r.pvalue] for r in [stats.ks_2samp(train[c], test[c]) for c in cols]], index=cols, columns=["KS_stat", "p_value"]).round(3)
ks["similar (p > 0.05)"] = ks["p_value"] > 0.05
ks

,KS_stat,p_value,similar (p > 0.05)
crosswind_ms,0.024,0.741,True
height_m,0.038,0.207,True
swath_m,0.047,0.055,True
temp_c,0.028,0.529,True
rh_pct,0.039,0.173,True
droplet_code,0.028,0.563,True
drift_distance_m,0.036,0.247,True


## M2 – Baseline models
**What:** score two simple models – an average guess and linear regression.

**Why:** they give the minimum score the main model must beat.
Scores: R² (share of drift variation explained, 1 = perfect), MAE (average error in metres), RMSE (error that punishes big mistakes).

In [4]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

results = []
def evaluate(name, model): model.fit(X_train, y_train); pred = model.predict(X_test); r2 = r2_score(y_test, pred); mae = mean_absolute_error(y_test, pred); rmse = np.sqrt(mean_squared_error(y_test, pred)); results.append({"model": name, "R2": round(r2, 3), "MAE_m": round(mae, 2), "RMSE_m": round(rmse, 2)}); print(f"{name:20s} R² = {r2:.3f} | MAE = {mae:.2f} m | RMSE = {rmse:.2f} m"); return pred

In [5]:
pred_avg = evaluate("Average guess", DummyRegressor(strategy="mean"))
pred_lin = evaluate("Linear regression", LinearRegression())

Average guess        R² = -0.001 | MAE = 9.17 m | RMSE = 11.47 m
Linear regression    R² = 0.839 | MAE = 3.76 m | RMSE = 4.60 m


**Findings:** The average-guess baseline explains none of the variation (R² ≈ 0, MAE 9.17 m). Linear regression
reaches R² = 0.839 (MAE 3.76 m, RMSE 4.60 m), already above the project target of 0.7. However, a linear model
gives each input the same effect for every droplet size, so it cannot capture the droplet × crosswind
interaction found in the EDA. The main model must clearly beat R² = 0.839.

## M3 – Random Forest
**What:** train a Random Forest (300 decision trees whose predictions are averaged).
**Why:** it can learn non-linear effects and interactions (e.g. wind matters more for fine droplets), which the linear model cannot.

In [6]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
pred_rf = evaluate("Random Forest", rf)

Random Forest        R² = 0.990 | MAE = 0.83 m | RMSE = 1.17 m


In [7]:
pd.DataFrame(results).set_index("model")

,R2,MAE_m,RMSE_m
model,,,
Average guess,-0.001,9.17,11.47
Linear regression,0.839,3.76,4.60
Random Forest,0.990,0.83,1.17


**Findings:** The Random Forest reaches R² = 0.990 on the test set (MAE 0.83 m, RMSE 1.17 m), compared with 0.839
(MAE 3.76 m) for linear regression – the average error is about 4.5× smaller. This confirms that the non-linear
effects and the droplet × crosswind interaction matter. The very high accuracy partly reflects that the data
come from a deterministic simulation without field noise; real-field accuracy will be lower and will be checked
with Suleco flight data (Dataset B).

In [8]:
train_r2 = r2_score(y_train, rf.predict(X_train))
print(f"Train R² = {train_r2:.3f} | Test R² = {r2_score(y_test, pred_rf):.3f}")

Train R² = 0.998 | Test R² = 0.990


In [9]:
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
X_all, y_all = df[FEATURES], df[TARGET]
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_r2 = cross_val_score(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1), X_all, y_all, cv=cv, scoring="r2")
cv_mae = -cross_val_score(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1), X_all, y_all, cv=cv, scoring="neg_mean_absolute_error")
print("R² per fold:", cv_r2.round(3), "| mean =", round(cv_r2.mean(), 3), "±", round(cv_r2.std(), 3))
print("MAE per fold:", cv_mae.round(2), "| mean =", round(cv_mae.mean(), 2), "m")

R² per fold: [0.989 0.987 0.988 0.99  0.987] | mean = 0.988 ± 0.001
MAE per fold: [0.84 0.86 0.9  0.85 0.9 ] | mean = 0.87 m


In [10]:
gcv = GroupKFold(n_splits=5)
g_r2 = cross_val_score(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1), X_all, y_all, cv=gcv, groups=df["datetime"], scoring="r2")
print("Grouped R² per fold:", g_r2.round(3), "| mean =", round(g_r2.mean(), 3))

Grouped R² per fold: [0.99  0.988 0.99  0.99  0.986] | mean = 0.989
